# ARC-AGI-3 - MuZero TTT at Deployment (Stage 3)

Loads the Stage 2 checkpoint and applies two-scope test-time training during
live ARC-AGI-3 play.

**Key constraints:**
- Scope 2 runs **before** each level using all prior-level experience
- Scope 1 runs **after** each level using only that level's experience
- Scope 1 LR = `3e-5`, Scope 2 LR = `1e-5`
- MCTS simulations = `25`, max moves = `80`
- Model stays a small ResNet: `3` blocks, `64` channels
- **NO color jitter** - ARC colors are semantic object identifiers
- If `stage2_finetune.checkpoint` is missing, the notebook falls back to
  random weights and keeps running in clear dry-run mode

**Input:** `stage2_finetune.checkpoint`
**Output:** per-game Stage 3 TTT-adapted checkpoints under `/kaggle/working/`


In [ ]:
import os, sys, subprocess, pathlib, datetime, pickle, copy, random, time

KAGGLE_INPUT = '/kaggle/input'
KAGGLE_WORK = '/kaggle/working'

# ── Dataset-sourced paths (no git clone needed for private repo) ─────────────
# benhaslam/arc3-neurosym dataset contains arc3_game.py + data_pipeline.py
NEUROSYM_DIR = '/kaggle/input/arc3-neurosym'
STAGE2_CKPT  = '/kaggle/input/arc3-stage2-checkpoint/stage2_finetune_2026-04-08--22-03-52.checkpoint'

# muzero-general is a public repo — cloned fresh each run
MUZERO_DIR = '/kaggle/working/muzero-general'

if not os.path.exists(MUZERO_DIR):
    subprocess.run([
        'git', 'clone', '--depth=1',
        'https://github.com/werner-duvaud/muzero-general.git',
        MUZERO_DIR,
    ], check=True)

subprocess.run([
    'pip', 'install', '-q',
    'torch', 'torchvision', 'ray[default]', 'tensorboard', 'Pillow', 'numpy', 'arc-agi'
], check=True)

for p in [MUZERO_DIR, NEUROSYM_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

ARC_API_KEY = os.environ.get('ARC_API_KEY')
if ARC_API_KEY is None:
    print('WARNING: ARC_API_KEY is not set. Live deployment cells will skip environment play.')
else:
    print('ARC_API_KEY loaded from Kaggle Secrets.')

import numpy as np
import torch

try:
    from arc_agi import Arcade
except ImportError:
    from arcengine import Arcade

from arcengine import GameAction, GameState

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Setup complete.')
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB)')
print(f'  neurosym: {NEUROSYM_DIR}')
print(f'  stage2_ckpt exists: {os.path.exists(STAGE2_CKPT)}')


In [ ]:
import models
from arc3_game import MuZeroConfig, TTTScope1, TTTScope2

BASE_CONFIG = MuZeroConfig(game_id='stage3', n_bins=4)
BASE_CONFIG.action_space = list(range(18))
BASE_CONFIG.observation_shape = (1, 64, 64)
BASE_CONFIG.stacked_observations = 0
BASE_CONFIG.max_moves = 80
BASE_CONFIG.num_simulations = 25
BASE_CONFIG.network = 'resnet'
BASE_CONFIG.downsample = 'CNN'
BASE_CONFIG.blocks = 3
BASE_CONFIG.channels = 64
BASE_CONFIG.train_on_gpu = torch.cuda.is_available()
BASE_CONFIG.results_path = pathlib.Path(KAGGLE_WORK) / 'results' / 'ttt' / \
    datetime.datetime.now().strftime('%Y-%m-%d--%H-%M-%S')
BASE_CONFIG.results_path.mkdir(parents=True, exist_ok=True)

STAGE2_PAYLOAD = None
STAGE2_WEIGHTS = None
STAGE2_CONFIG = None
DRY_RUN = False

if os.path.exists(STAGE2_CKPT):
    with open(STAGE2_CKPT, 'rb') as f:
        STAGE2_PAYLOAD = pickle.load(f)
    STAGE2_WEIGHTS = STAGE2_PAYLOAD.get('weights')
    STAGE2_CONFIG = STAGE2_PAYLOAD.get('config')
    print('Stage 2 checkpoint loaded.')
    print(f"  stage: {STAGE2_PAYLOAD.get('stage', 'unknown')}")
    print(f"  training_steps: {STAGE2_PAYLOAD.get('training_steps', 'unknown')}")
    if STAGE2_CONFIG is not None:
        for attr in [
            'observation_shape', 'stacked_observations', 'support_size',
            'network', 'downsample', 'blocks', 'channels',
            'reduced_channels_reward', 'reduced_channels_value', 'reduced_channels_policy',
            'resnet_fc_reward_layers', 'resnet_fc_value_layers', 'resnet_fc_policy_layers',
            'discount', 'value_loss_weight', 'weight_decay'
        ]:
            if hasattr(STAGE2_CONFIG, attr):
                setattr(BASE_CONFIG, attr, getattr(STAGE2_CONFIG, attr))
else:
    DRY_RUN = True
    print('WARNING: stage2_finetune.checkpoint not found. Using random weights in dry-run mode.')

BASE_MODEL = models.MuZeroNetwork(BASE_CONFIG).to(DEVICE)
BASE_MODEL.ttt_config = BASE_CONFIG
if STAGE2_WEIGHTS is not None:
    BASE_MODEL.set_weights(STAGE2_WEIGHTS)
BASE_WEIGHTS = copy.deepcopy(BASE_MODEL.get_weights())

print('Stage 3 base config ready.')
print(f'  device          : {DEVICE}')
print(f'  policy dim      : {len(BASE_CONFIG.action_space)}')
print(f'  support size    : {BASE_CONFIG.support_size}')
print(f'  num simulations : {BASE_CONFIG.num_simulations}')
print(f'  dry run         : {DRY_RUN}')


In [ ]:
def _one_hot_policy(action_batch: torch.Tensor, policy_dim: int) -> torch.Tensor:
    target = torch.zeros(action_batch.shape[0], policy_dim, device=action_batch.device)
    target.scatter_(1, action_batch.long(), 1.0)
    return target


def run_ttt_update(model, optimizer, experience_buffer, n_steps, lr):
    '''
    Run in-place TTT on one-step experience tuples:
      (obs, action, reward, next_obs)

    Loss mirrors the muzero-general update path:
    - support-based value loss
    - support-based reward loss
    - policy cross-entropy loss
    '''
    if n_steps <= 0 or not experience_buffer:
        return

    config = getattr(model, 'ttt_config', BASE_CONFIG)
    device = next(model.parameters()).device
    support_size = config.support_size
    discount = config.discount
    value_loss_weight = config.value_loss_weight
    batch_size = min(32, len(experience_buffer))
    was_training = model.training
    model.train()

    for group in optimizer.param_groups:
        group['lr'] = lr

    for _ in range(n_steps):
        if len(experience_buffer) >= batch_size:
            batch = random.sample(experience_buffer, k=batch_size)
        else:
            batch = [random.choice(experience_buffer) for _ in range(batch_size)]

        obs = torch.tensor(
            np.stack([item[0] for item in batch]), dtype=torch.float32, device=device
        )
        action = torch.tensor(
            [item[1] for item in batch], dtype=torch.long, device=device
        ).unsqueeze(-1)
        reward = torch.tensor(
            [[item[2]] for item in batch], dtype=torch.float32, device=device
        )
        next_obs = torch.tensor(
            np.stack([item[3] for item in batch]), dtype=torch.float32, device=device
        )

        with torch.no_grad():
            next_value_logits, _, next_policy_logits, _ = model.initial_inference(next_obs)
            next_value_scalar = models.support_to_scalar(next_value_logits, support_size)
            next_policy_target = torch.softmax(next_policy_logits, dim=1)

        root_value_target = reward + discount * next_value_scalar
        root_value_target = models.scalar_to_support(root_value_target, support_size).squeeze(1)
        recurrent_value_target = models.scalar_to_support(next_value_scalar, support_size).squeeze(1)
        recurrent_reward_target = models.scalar_to_support(reward, support_size).squeeze(1)

        value0, reward0, policy0, hidden_state = model.initial_inference(obs)
        if hidden_state.requires_grad:
            hidden_state.register_hook(lambda grad: grad * 0.5)
        value1, reward1, policy1, _ = model.recurrent_inference(hidden_state, action)

        policy_target_root = _one_hot_policy(action, policy0.shape[1])

        value_loss0 = (-root_value_target * torch.nn.LogSoftmax(dim=1)(value0)).sum(1)
        policy_loss0 = (-policy_target_root * torch.nn.LogSoftmax(dim=1)(policy0)).sum(1)

        value_loss1 = (-recurrent_value_target * torch.nn.LogSoftmax(dim=1)(value1)).sum(1)
        reward_loss1 = (-recurrent_reward_target * torch.nn.LogSoftmax(dim=1)(reward1)).sum(1)
        policy_loss1 = (-next_policy_target * torch.nn.LogSoftmax(dim=1)(policy1)).sum(1)

        loss = (
            value_loss_weight * (value_loss0 + value_loss1)
            + reward_loss1
            + policy_loss0
            + policy_loss1
        ).mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

    model.train(was_training)


print('run_ttt_update(model, optimizer, experience_buffer, n_steps, lr) is ready.')


In [ ]:
from self_play import MCTS


def select_action(model, obs, config, temperature=1.0):
    legal_actions = list(getattr(config, 'legal_actions', config.action_space))
    if not legal_actions:
        raise ValueError('No legal actions provided to select_action.')

    was_training = model.training
    model.eval()
    with torch.no_grad():
        root, _ = MCTS(config).run(
            model,
            obs,
            legal_actions,
            to_play=0,
            add_exploration_noise=False,
        )

    visit_counts = np.array(
        [root.children[a].visit_count if a in root.children else 0 for a in legal_actions],
        dtype=np.float32,
    )

    if temperature <= 1e-6 or np.all(visit_counts == 0):
        action = legal_actions[int(np.argmax(visit_counts))]
    else:
        scaled = np.power(np.maximum(visit_counts, 1e-8), 1.0 / temperature)
        scaled_sum = scaled.sum()
        if not np.isfinite(scaled_sum) or scaled_sum <= 0:
            action = legal_actions[int(np.argmax(visit_counts))]
        else:
            probs = scaled / scaled_sum
            action = int(np.random.choice(legal_actions, p=probs))

    model.train(was_training)
    return int(action)


print('select_action(model, obs, config, temperature=1.0) is ready.')


In [ ]:
from types import SimpleNamespace


def normalize_obs(frame):
    arr = np.array(frame, dtype=np.float32)
    if arr.ndim == 2:
        arr = arr[np.newaxis, ...]
    return arr / 12.0


def legal_actions_for_state(state, n_bins=4):
    available_actions = list(getattr(state, 'available_actions', []) or [])
    if available_actions == [6]:
        return list(range(n_bins * n_bins)), True
    return [int(a) - 1 for a in available_actions], False


def decode_arc_action(action_int, available_actions, n_bins=4):
    if list(available_actions) == [6]:
        bin_x = int(action_int % n_bins)
        bin_y = int(action_int // n_bins)
        # bin centre in [0, 64) — correct for any n_bins value
        px = int((bin_x + 0.5) * 64 / n_bins)
        py = int((bin_y + 0.5) * 64 / n_bins)
        action = GameAction.from_id(6)
        try:
            action.set_data({'x': px, 'y': py})
        except Exception:
            pass
        return action, {'x': px, 'y': py}

    action_id = int(action_int) + 1
    action = GameAction.from_id(action_id)
    return action, {}


def shaped_reward(prev_state, next_state):
    reward = -0.01
    prev_levels = int(getattr(prev_state, 'levels_completed', 0) or 0)
    next_levels = int(getattr(next_state, 'levels_completed', 0) or 0)
    delta = next_levels - prev_levels
    if delta > 0:
        reward += float(delta)
    win_levels = int(getattr(next_state, 'win_levels', 0) or 0)
    if getattr(next_state, 'state', None) == GameState.WIN or (win_levels > 0 and next_levels >= win_levels):
        reward += 10.0
    return float(reward)


GAME_IDS = ['ls20', 'lp85', 'ft09', 'vc33']
adapted_checkpoints = {}
deployment_logs = {}

if ARC_API_KEY is None:
    print('WARNING: ARC_API_KEY missing. Skipping live deployment and saving untouched base weights.')
    for game_id in GAME_IDS:
        search_config = copy.deepcopy(BASE_CONFIG)
        search_config.game_id = game_id
        adapted_checkpoints[game_id] = {
            'weights': copy.deepcopy(BASE_WEIGHTS),
            'config': search_config,
            'reports': [{'skipped': 'missing ARC_API_KEY'}],
            'dry_run': DRY_RUN,
        }
        deployment_logs[game_id] = adapted_checkpoints[game_id]['reports']
else:
    for game_id in GAME_IDS:
        print(f'\n=== {game_id} ===')

        model = models.MuZeroNetwork(BASE_CONFIG).to(DEVICE)
        model.set_weights(copy.deepcopy(BASE_WEIGHTS))
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=3e-5,
            weight_decay=getattr(BASE_CONFIG, 'weight_decay', 1e-4),
        )

        search_config = copy.deepcopy(BASE_CONFIG)
        search_config.game_id = game_id
        search_config.max_moves = 80
        search_config.num_simulations = 25

        scope_state = SimpleNamespace(level_history=[])
        scope1 = TTTScope1(scope_state)
        scope2 = TTTScope2(scope_state)
        scope2_buffer = []
        level_reports = []

        try:
            arcade = Arcade()
            env = arcade.make(game_id)
            state = env.reset()
            obs = normalize_obs(state.frame)
            available_actions = list(getattr(state, 'available_actions', []) or [])
            initial_legal_actions, _ = legal_actions_for_state(state, n_bins=4)
            if initial_legal_actions:
                search_config.action_space = initial_legal_actions[:]
                search_config.legal_actions = initial_legal_actions[:]
            model.ttt_config = search_config

            total_steps = 0
            done = False
            current_level = max(1, int(getattr(state, 'levels_completed', 0) or 0) + 1)

            while not done and total_steps < search_config.max_moves:
                scope2_budget = scope2.ttt_budget(current_level)
                run_ttt_update(model, optimizer, scope2_buffer, scope2_budget, lr=1e-5)

                experience_buffer = []
                level_steps = 0
                level_reward = 0.0
                level_start = current_level

                while not done and total_steps < search_config.max_moves:
                    available_actions = list(getattr(state, 'available_actions', []) or [])
                    legal_actions, _ = legal_actions_for_state(state, n_bins=4)
                    if not legal_actions:
                        print(f'  level {level_start:02d} | no legal actions exposed; stopping early')
                        done = True
                        break

                    search_config.action_space = legal_actions[:]
                    search_config.legal_actions = legal_actions[:]
                    action_int = select_action(model, obs, search_config, temperature=1.0)
                    action_obj, action_data = decode_arc_action(action_int, available_actions, n_bins=4)
                    if action_data:
                        next_state = env.step(action_obj, data=action_data)
                    else:
                        next_state = env.step(action_obj)
                    next_obs = normalize_obs(next_state.frame)
                    reward = shaped_reward(state, next_state)

                    experience_buffer.append((
                        obs.astype(np.float32),
                        int(action_int),
                        float(reward),
                        next_obs.astype(np.float32),
                    ))

                    prev_levels = int(getattr(state, 'levels_completed', 0) or 0)
                    next_levels = int(getattr(next_state, 'levels_completed', 0) or 0)
                    win_levels = int(getattr(next_state, 'win_levels', 0) or 0)
                    level_complete = next_levels > prev_levels
                    terminal = getattr(next_state, 'state', None) in (GameState.GAME_OVER, GameState.WIN)
                    game_won = win_levels > 0 and next_levels >= win_levels

                    obs = next_obs
                    state = next_state
                    total_steps += 1
                    level_steps += 1
                    level_reward += reward

                    if level_complete:
                        scope_state.level_history.append({
                            'step': total_steps,
                            'level_completed': next_levels,
                            'reward': float(level_reward),
                        })
                        current_level = next_levels + 1
                        break

                    if terminal or game_won:
                        done = True
                        break

                scope1_budget = scope1.ttt_budget(level_start)
                run_ttt_update(model, optimizer, experience_buffer, scope1_budget, lr=3e-5)
                scope2_buffer.extend(experience_buffer)

                report = {
                    'level': level_start,
                    'steps': level_steps,
                    'reward': float(level_reward),
                    'scope2_budget': scope2_budget,
                    'scope1_budget': scope1_budget,
                    'scope2_buffer_before_level': len(scope2_buffer) - len(experience_buffer),
                    'buffer_size_after_level': len(scope2_buffer),
                }
                level_reports.append(report)
                print(
                    f"  level {level_start:02d} | steps={level_steps:02d} | "
                    f"reward={level_reward:6.2f} | scope2={scope2_budget:2d} | scope1={scope1_budget:2d}"
                )

                if done:
                    break

        except Exception as exc:
            level_reports.append({'error': repr(exc)})
            print(f'  ERROR during {game_id}: {exc}')

        adapted_checkpoints[game_id] = {
            'weights': copy.deepcopy(model.get_weights()),
            'config': search_config,
            'reports': level_reports,
            'dry_run': DRY_RUN,
        }
        deployment_logs[game_id] = level_reports

print('\nStage 3 competition loop complete.')


In [ ]:
output_paths = {}

for game_id, payload in adapted_checkpoints.items():
    out_path = pathlib.Path(KAGGLE_WORK) / f'{game_id}_stage3_ttt.checkpoint'
    with open(out_path, 'wb') as f:
        pickle.dump({
            'weights': payload['weights'],
            'config': payload['config'],
            'stage': 3,
            'game_id': game_id,
            'dry_run': payload.get('dry_run', DRY_RUN),
            'stage2_ckpt': STAGE2_CKPT,
            'reports': payload.get('reports', []),
            'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
        }, f)
    output_paths[game_id] = str(out_path)
    print(f'Saved {game_id}: {out_path}')

print('\nArtifacts ready under /kaggle/working/:')
for game_id, path in output_paths.items():
    print(f'  {game_id}: {path}')


## Next steps

- Compare Stage 2 vs Stage 3 scorecards per game and log where Scope 2 helps
- Add a legality-aware future-action mask inside MuZero's internal rollouts so
  imagined nodes do not spend probability mass on impossible actions
- Export per-level TTT diagnostics (loss before/after each scope, budget used,
  value calibration drift)
- Run ablations: Scope 1 only, Scope 2 only, both scopes, and no TTT
- If level 7+ budgets stay at `2 + 1 = 3` steps as expected, treat Stage 3 as
  near-inference adaptation rather than another training phase
